In [1]:
!pip -q install transformers scipy soundfile

In [2]:
import torch
import numpy as np
import scipy.io.wavfile as wavfile

from transformers import (
    AutoProcessor,
    MusicgenForConditionalGeneration
)
from IPython.display import Audio, display
from google.colab import files

MODEL_NAME = "facebook/musicgen-small"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = MusicgenForConditionalGeneration.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

print("Pretrained MusicGen model loaded successfully!")

Device: cuda


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.87k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

Pretrained MusicGen model loaded successfully!


In [3]:
prompt = (
    "Relaxing instrumental music with peaceful piano, "
    "soft flute, gentle melody and a calm atmosphere"
)

inputs = processor(
    text=[prompt],
    padding=True,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    audio_values = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True
    )

audio = audio_values[0, 0].cpu().numpy()

sample_rate = model.config.audio_encoder.sampling_rate

print("Music generated successfully!")
print("Sample rate:", sample_rate)
print("Audio duration:", round(len(audio) / sample_rate, 2), "seconds")

Music generated successfully!
Sample rate: 32000
Audio duration: 5.06 seconds


In [4]:
display(Audio(audio, rate=sample_rate))

In [5]:
audio_int16 = np.int16(
    np.clip(audio, -1.0, 1.0) * 32767
)

output_file = "generated_music.wav"

wavfile.write(
    output_file,
    sample_rate,
    audio_int16
)

print("Saved:", output_file)

files.download(output_file)

Saved: generated_music.wav


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>